In [3]:
# 1) Import required libraries
import pandas as pd
import numpy as np

# 2) Load the CSV file into a DataFrame named df
file_path = "../data/raw/DISSANAYAKA POS DATASET.csv"
df = pd.read_csv(file_path)

# 3) Display the first 5 rows
display(df.head())

# 4) Print the shape of the DataFrame
print(f"DataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns")

# 5) Print total missing/null values for each column
print("\nMissing/null values per column:")
print(df.isnull().sum())

# 6) Print data types of all columns
print("\nData types of all columns:")
print(df.dtypes)

,TransactionID,Date,Time,ProductID,ProductName,Category,PricingUnit,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,Payment Method
0,TX0005400000,1/1/2021,7:47:47,PI00469,Whiskas Adult Ocean Fish Food - 500g,Pet Products,Pack,820,700,820,1,970,Cash
1,TX0005400000,1/1/2021,7:47:47,PI00097,Kotmale Milk Chocolate - 170ml,Dairy,Pack,50,45,150,3,970,Cash
2,TX0005400001,1/1/2021,8:08:33,PI03359,Precare Unisex Sandal 1 S07 Bj0003 - 1pc,Health & Beauty,Unit,5500,4600,5500,1,5980,Cash
3,TX0005400001,1/1/2021,8:08:33,PI03268,Newtro Chili Powder - 100g,Seeds & Spices,Pack,190,170,380,2,5980,Cash
4,TX0005400001,1/1/2021,8:08:33,PI00173,Denta Comfort Soft Toothbrushes - 2pcs,Health & Beauty,Unit,100,80,100,1,5980,Cash


DataFrame shape: 812258 rows, 13 columns

Missing/null values per column:
TransactionID     0
Date              0
Time              0
ProductID         0
ProductName       0
Category          0
PricingUnit       0
UnitPrice         0
BuyingPrice       0
SellingPrice      0
Quantity          0
Total_LKR         0
Payment Method    0
dtype: int64

Data types of all columns:
TransactionID     object
Date              object
Time              object
ProductID         object
ProductName       object
Category          object
PricingUnit       object
UnitPrice          int64
BuyingPrice        int64
SellingPrice       int64
Quantity           int64
Total_LKR          int64
Payment Method    object
dtype: object


In [4]:
# Data Cleaning and Preprocessing on existing DataFrame: df

import os
import numpy as np
import pandas as pd

# Work on a copy to keep original df unchanged
df_work = df.copy()

# 1) Fill missing values in numerical columns with median
numerical_cols = df_work.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df_work[col].isnull().any():
        median_value = df_work[col].median()
        if pd.isna(median_value):
            median_value = 0
        df_work[col] = df_work[col].fillna(median_value)

# 2) Fill missing values in categorical columns with mode
categorical_cols = df_work.select_dtypes(include=["object", "category"]).columns.tolist()
for col in categorical_cols:
    if df_work[col].isnull().any():
        mode_series = df_work[col].mode(dropna=True)
        mode_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
        df_work[col] = df_work[col].fillna(mode_value)

# 3) Drop duplicate rows
duplicates_removed = int(df_work.duplicated().sum())
df_work = df_work.drop_duplicates().reset_index(drop=True)

# 4) Memory-safe categorical encoding
max_onehot_unique = 30
low_card_cols = [c for c in categorical_cols if df_work[c].nunique(dropna=False) <= max_onehot_unique]
high_card_cols = [c for c in categorical_cols if c not in low_card_cols]

# One-hot encode only low-cardinality categorical columns
if low_card_cols:
    df_cleaned = pd.get_dummies(df_work, columns=low_card_cols, drop_first=True, dtype="uint8")
else:
    df_cleaned = df_work.copy()

# Encode high-cardinality columns with compact integer codes
for col in high_card_cols:
    df_cleaned[col] = df_cleaned[col].astype("category").cat.codes.astype("int32")

# Optional numeric downcasting to reduce memory
float_cols = df_cleaned.select_dtypes(include=["float64"]).columns
df_cleaned[float_cols] = df_cleaned[float_cols].astype("float32")

int_cols = df_cleaned.select_dtypes(include=["int64"]).columns
df_cleaned[int_cols] = df_cleaned[int_cols].astype("int32")

# 5) Ensure output directory exists
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# 6) Save cleaned dataset without index
output_path = os.path.join(output_dir, "cleaned_dataset.csv")
df_cleaned.to_csv(output_path, index=False)

# 7) Print summary and preview
print(f"Success: Cleaned dataset saved to {output_path}")
print(f"Duplicate rows removed: {duplicates_removed}")
print(f"Low-cardinality columns one-hot encoded: {len(low_card_cols)}")
print(f"High-cardinality columns label-encoded: {len(high_card_cols)}")
print(f"Cleaned dataframe shape: {df_cleaned.shape}")
display(df_cleaned.head())

Success: Cleaned dataset saved to ../data/processed\cleaned_dataset.csv
Duplicate rows removed: 0
Low-cardinality columns one-hot encoded: 3
High-cardinality columns label-encoded: 5
Cleaned dataframe shape: (812258, 36)


,TransactionID,Date,Time,ProductID,ProductName,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,...,Category_Seafood,Category_Seeds & Spices,Category_Snacks & Confectionery,Category_Stationery,Category_Tea & Coffee,Category_Vegetables,PricingUnit_Unit,PricingUnit_g,PricingUnit_kg,Payment Method_Cash
0,0,0,39356,468,3765,820,700,820,1,970,...,0,0,0,0,0,0,0,0,0,1
1,0,0,39356,96,1679,50,45,150,3,970,...,0,0,0,0,0,0,0,0,0,1
2,1,0,40325,3358,2849,5500,4600,5500,1,5980,...,0,0,0,0,0,0,1,0,0,1
3,1,0,40325,3267,2588,190,170,380,2,5980,...,0,1,0,0,0,0,0,0,0,1
4,1,0,40325,172,776,100,80,100,1,5980,...,0,0,0,0,0,0,1,0,0,1


In [5]:
# Most likely target for retail demand forecasting from your columns is "Quantity"
# (columns observed include: TransactionID, Date, Time, ProductID, ProductName, Category,
# PricingUnit, UnitPrice, BuyingPrice, SellingPrice, Quantity, Total_LKR, Payment Method)

from sklearn.model_selection import train_test_split

# Use the already-cleaned dataframe in memory
# Prefer df_cleaned if it exists; otherwise use df
data = df_cleaned.copy() if "df_cleaned" in globals() else df.copy()

# Identified target column
target_col = "Quantity"

# Separate features (X) and target (y)
X = data.drop(columns=[target_col])
y = data[target_col]

# Train-test split: 80% train, 20% test, fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Verify split result
print(f"Target column used: {target_col}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Target column used: Quantity
X_train shape: (649806, 35)
X_test shape: (162452, 35)
y_train shape: (649806,)
y_test shape: (162452,)


In [6]:
# 1) Import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

# 2) Import evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 3) Import joblib and os for model saving
import joblib
import os
import numpy as np

# 4) Initialize the regression model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 5) Train the model on training data
model.fit(X_train, y_train)

# 6) Make predictions on the test set
y_pred = model.predict(X_test)

# 7) Calculate and print MAE, RMSE, and R2 Score
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")

# 8) Ensure ../models/ directory exists and save the trained model
model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "retail_rf_model.pkl")
joblib.dump(model, model_path)

# 9) Print success message with model save location
print(f"Model saved successfully at: {model_path}")

MAE  : 0.0001
RMSE : 0.0050
R2   : 1.0000
Model saved successfully at: ../models\retail_rf_model.pkl


In [7]:
# 1) Load the saved Random Forest model
import joblib

model_path = "../models/retail_rf_model.pkl"
loaded_model = joblib.load(model_path)

# 2) Take one sample row from X_test to simulate new unseen input
sample_input = X_test.iloc[[0]]

# 3) Predict demand/sales for this sample
predicted_value = loaded_model.predict(sample_input)[0]

# 4) Get the corresponding actual value from y_test
actual_value = y_test.iloc[0]

# 5) Print actual vs predicted in a clear format
print(f"Actual Sales   : {actual_value:.4f}")
print(f"Predicted Sales: {predicted_value:.4f}")

Actual Sales   : 2.0000
Predicted Sales: 2.0000
